# OP3 bootstrap comparisons


# 1. Settings


In [1]:
print(1)

1


In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from op3_bootstrap import (
    method_roots, load_scores, paired_results, bootstrap_summary,
    save_summary, readable_summary,
)

CASES = [
    ("full_coverage", "main"),
    ("coverage_match", "main"),
    ("full_coverage", "ablation"),
    ("coverage_match", "ablation"),
]
SEEDS = [0, 1, 2, 3, 4, 10, 11, 12, 13, 14]
N_BOOT = 2000
BOOTSTRAP_SEED = 20260505
CI_LEVEL = 0.95

display(pd.DataFrame(CASES, columns=["coverage", "analysis"]))


,coverage,analysis
0,full_coverage,main
1,coverage_match,main
2,full_coverage,ablation
3,coverage_match,ablation


# 2. Bootstrap


In [3]:
case_results = {}

for coverage, analysis in CASES:
    print(f"\n{coverage} / {analysis}")
    roots = method_roots(PROJECT_ROOT, coverage, analysis)

    scores, compounds, cell_types, metric_check = load_scores(roots, SEEDS)
    unit_deltas, compound_wins = paired_results(scores, compounds, cell_types, roots)
    summary = bootstrap_summary(
        unit_deltas, compound_wins, seeds=SEEDS,
        n_boot=N_BOOT, bootstrap_seed=BOOTSTRAP_SEED, ci_level=CI_LEVEL,
    )
    path = save_summary(PROJECT_ROOT, coverage, analysis, summary)
    print(path.relative_to(PROJECT_ROOT))

    case_results[(coverage, analysis)] = {
        "summary": summary,
        "unit_deltas": unit_deltas,
        "compound_wins": compound_wins,
        "metric_check": metric_check,
    }



full_coverage / main
Loading none
Loading lpm_trainable_all_last_full_cov_stepmatched
Loading lpm_fixed_all_last_full_cov_stepmatched
Loading lpm_trainable_l1000_last_stepmatched
Loading lpm_fixed_l1000_last_stepmatched
Loading lpm_trainable_all_20
Loading lpm_fixed_all_20
Loading fp_trainable_fp
Loading lpm_trainable_l1000_20
Loading lpm_fixed_l1000_20
Loading fp_fixed_fp
Loading chemb1_fixed_chemb1
Loading chemb1_trainable_chemb1
Loading chemb2_fixed_chemb2
Loading chemb2_trainable_chemb2
Loading molformer_fixed_molformer
Loading molformer_trainable_molformer
Loading lpm_trainable_all_20_narrowed
Loading lpm_fixed_all_20_narrowed
All 760 metric checks passed; maximum difference 4.97e-11
results/df_bootstrap_summary_full_coverage.csv

coverage_match / main
Loading none
Loading lpm_trainable_all_last_stepmatched_coverage_match
Loading lpm_fixed_all_last_stepmatched_coverage_match
Loading lpm_trainable_l1000_last_stepmatched
Loading lpm_fixed_l1000_last_stepmatched
Loading lpm_trainabl

# 3. Tables


In [4]:
for (coverage, analysis), result in case_results.items():
    print(f"\n{coverage} / {analysis}")
    display(readable_summary(result["summary"]))



full_coverage / main


,method,spearman delta [95% CI],spearman win rate [95% CI],cosine delta [95% CI],cosine win rate [95% CI],mrrmse delta [95% CI],mrrmse win rate [95% CI],mae delta [95% CI],mae win rate [95% CI]
0,chemb1_fixed_chemb1,"-0.0724 [-0.1317, -0.0156]","0.297 [0.193, 0.434]","-0.0743 [-0.1329, -0.0152]","0.311 [0.207, 0.464]","-0.0581 [-0.1135, -0.0273]","0.282 [0.173, 0.414]","-0.0535 [-0.0939, -0.0288]","0.232 [0.139, 0.353]"
1,lpm_fixed_all_last_full_cov_stepmatched,"-0.0507 [-0.0961, -0.0100]","0.391 [0.282, 0.506]","-0.0490 [-0.0961, -0.0088]","0.377 [0.265, 0.500]","-0.0506 [-0.0993, -0.0226]","0.319 [0.217, 0.436]","-0.0493 [-0.0881, -0.0270]","0.319 [0.207, 0.443]"
2,molformer_fixed_molformer,"-0.0442 [-0.0964, 0.0046]","0.362 [0.246, 0.500]","-0.0396 [-0.0960, 0.0102]","0.348 [0.236, 0.457]","-0.0466 [-0.0904, -0.0182]","0.363 [0.254, 0.486]","-0.0493 [-0.0861, -0.0243]","0.341 [0.227, 0.457]"
3,chemb2_fixed_chemb2,"-0.0344 [-0.0817, 0.0179]","0.378 [0.264, 0.507]","-0.0312 [-0.0849, 0.0206]","0.384 [0.284, 0.507]","-0.0392 [-0.0715, -0.0140]","0.355 [0.236, 0.486]","-0.0409 [-0.0705, -0.0190]","0.312 [0.203, 0.436]"
4,fp_fixed_fp,"-0.0280 [-0.0798, 0.0092]","0.492 [0.358, 0.636]","-0.0271 [-0.0802, 0.0161]","0.477 [0.354, 0.601]","-0.0388 [-0.0811, -0.0054]","0.362 [0.240, 0.500]","-0.0418 [-0.0756, -0.0195]","0.282 [0.188, 0.413]"
5,chemb2_trainable_chemb2,"-0.0197 [-0.0562, 0.0189]","0.449 [0.333, 0.562]","-0.0174 [-0.0574, 0.0250]","0.442 [0.322, 0.557]","-0.0300 [-0.0612, -0.0055]","0.370 [0.250, 0.494]","-0.0330 [-0.0591, -0.0152]","0.341 [0.210, 0.472]"
6,lpm_fixed_l1000_20,"-0.0163 [-0.0533, 0.0170]","0.449 [0.336, 0.574]","-0.0145 [-0.0507, 0.0183]","0.463 [0.348, 0.579]","-0.0294 [-0.0707, -0.0053]","0.463 [0.333, 0.585]","-0.0299 [-0.0667, -0.0103]","0.406 [0.292, 0.537]"
7,lpm_fixed_all_20_narrowed,"-0.0314 [-0.0596, -0.0011]","0.441 [0.321, 0.560]","-0.0299 [-0.0604, 0.0008]","0.434 [0.316, 0.563]","-0.0280 [-0.0583, -0.0085]","0.384 [0.266, 0.521]","-0.0315 [-0.0582, -0.0154]","0.333 [0.214, 0.473]"
8,lpm_fixed_all_20,"+0.0007 [-0.0305, 0.0358]","0.500 [0.371, 0.629]","+0.0032 [-0.0299, 0.0384]","0.521 [0.400, 0.643]","-0.0275 [-0.0590, 0.0013]","0.399 [0.278, 0.529]","-0.0381 [-0.0659, -0.0171]","0.319 [0.203, 0.449]"
9,molformer_trainable_molformer,"-0.0187 [-0.0515, 0.0192]","0.471 [0.353, 0.587]","-0.0150 [-0.0520, 0.0222]","0.492 [0.369, 0.622]","-0.0241 [-0.0534, -0.0022]","0.442 [0.321, 0.573]","-0.0257 [-0.0502, -0.0085]","0.427 [0.300, 0.552]"



coverage_match / main


,method,spearman delta [95% CI],spearman win rate [95% CI],cosine delta [95% CI],cosine win rate [95% CI],mrrmse delta [95% CI],mrrmse win rate [95% CI],mae delta [95% CI],mae win rate [95% CI]
0,chemb1_fixed_chemb1,"-0.0554 [-0.1132, -0.0044]","0.326 [0.207, 0.471]","-0.0537 [-0.1131, 0.0030]","0.347 [0.239, 0.500]","-0.0596 [-0.1146, -0.0275]","0.311 [0.203, 0.440]","-0.0592 [-0.1025, -0.0347]","0.232 [0.150, 0.340]"
1,lpm_fixed_all_last_stepmatched_coverage_match,"-0.0483 [-0.0909, -0.0109]","0.399 [0.279, 0.522]","-0.0452 [-0.0884, -0.0062]","0.384 [0.271, 0.521]","-0.0486 [-0.0964, -0.0232]","0.319 [0.207, 0.442]","-0.0483 [-0.0857, -0.0267]","0.319 [0.210, 0.443]"
2,molformer_fixed_molformer,"-0.0431 [-0.0962, 0.0056]","0.341 [0.228, 0.486]","-0.0384 [-0.0956, 0.0107]","0.370 [0.254, 0.486]","-0.0476 [-0.0919, -0.0202]","0.370 [0.257, 0.493]","-0.0495 [-0.0851, -0.0244]","0.348 [0.239, 0.464]"
3,chemb2_fixed_chemb2,"-0.0488 [-0.0942, 0.0063]","0.334 [0.232, 0.457]","-0.0453 [-0.0958, 0.0094]","0.370 [0.266, 0.491]","-0.0453 [-0.0761, -0.0213]","0.297 [0.195, 0.414]","-0.0461 [-0.0743, -0.0259]","0.239 [0.145, 0.356]"
4,fp_fixed_fp,"-0.0477 [-0.1111, -0.0054]","0.456 [0.329, 0.602]","-0.0483 [-0.1062, 0.0017]","0.449 [0.322, 0.584]","-0.0451 [-0.0874, -0.0168]","0.348 [0.229, 0.486]","-0.0447 [-0.0773, -0.0232]","0.283 [0.183, 0.407]"
5,chemb2_trainable_chemb2,"-0.0272 [-0.0636, 0.0127]","0.449 [0.332, 0.564]","-0.0249 [-0.0644, 0.0186]","0.435 [0.318, 0.551]","-0.0333 [-0.0651, -0.0117]","0.398 [0.275, 0.529]","-0.0347 [-0.0601, -0.0179]","0.340 [0.214, 0.471]"
6,lpm_fixed_all_20,"-0.0048 [-0.0372, 0.0276]","0.478 [0.350, 0.609]","-0.0021 [-0.0352, 0.0330]","0.500 [0.379, 0.630]","-0.0313 [-0.0606, -0.0025]","0.370 [0.257, 0.507]","-0.0409 [-0.0678, -0.0207]","0.276 [0.171, 0.400]"
7,lpm_fixed_l1000_20,"-0.0163 [-0.0533, 0.0170]","0.449 [0.336, 0.574]","-0.0145 [-0.0507, 0.0183]","0.463 [0.348, 0.579]","-0.0294 [-0.0707, -0.0053]","0.463 [0.333, 0.585]","-0.0299 [-0.0667, -0.0103]","0.406 [0.292, 0.537]"
8,lpm_fixed_all_20_narrowed,"-0.0314 [-0.0596, -0.0011]","0.441 [0.321, 0.560]","-0.0299 [-0.0604, 0.0008]","0.434 [0.316, 0.563]","-0.0280 [-0.0583, -0.0085]","0.384 [0.266, 0.521]","-0.0315 [-0.0582, -0.0154]","0.333 [0.214, 0.473]"
9,molformer_trainable_molformer,"-0.0161 [-0.0476, 0.0221]","0.450 [0.336, 0.566]","-0.0123 [-0.0488, 0.0247]","0.507 [0.393, 0.630]","-0.0249 [-0.0547, -0.0043]","0.493 [0.371, 0.617]","-0.0257 [-0.0522, -0.0082]","0.449 [0.326, 0.569]"



full_coverage / ablation


,method,spearman delta [95% CI],spearman win rate [95% CI],cosine delta [95% CI],cosine win rate [95% CI],mrrmse delta [95% CI],mrrmse win rate [95% CI],mae delta [95% CI],mae win rate [95% CI]
0,lpm_fixed_all_20_full_cov_abl_dose_time,"-0.0277 [-0.0801, 0.0247]","0.464 [0.341, 0.609]","-0.0241 [-0.0788, 0.0286]","0.464 [0.323, 0.613]","-0.0461 [-0.0904, -0.0134]","0.327 [0.215, 0.464]","-0.0500 [-0.0865, -0.0258]","0.283 [0.170, 0.420]"
1,lpm_fixed_all_20_full_cov_abl_dataset_dose_time,"-0.0416 [-0.0958, 0.0176]","0.406 [0.291, 0.539]","-0.0408 [-0.0983, 0.0200]","0.399 [0.279, 0.536]","-0.0334 [-0.0736, 0.0020]","0.320 [0.208, 0.472]","-0.0372 [-0.0730, -0.0129]","0.334 [0.217, 0.469]"
2,lpm_fixed_l1000_20,"-0.0163 [-0.0533, 0.0170]","0.449 [0.336, 0.574]","-0.0145 [-0.0507, 0.0183]","0.463 [0.348, 0.579]","-0.0294 [-0.0707, -0.0053]","0.463 [0.333, 0.585]","-0.0299 [-0.0667, -0.0103]","0.406 [0.292, 0.537]"
3,lpm_fixed_all_20_full_cov_abl_dataset,"-0.0245 [-0.0730, 0.0164]","0.435 [0.305, 0.565]","-0.0216 [-0.0697, 0.0232]","0.449 [0.321, 0.580]","-0.0284 [-0.0697, 0.0041]","0.399 [0.271, 0.537]","-0.0368 [-0.0708, -0.0105]","0.391 [0.264, 0.514]"
4,lpm_fixed_all_20_narrowed,"-0.0314 [-0.0596, -0.0011]","0.441 [0.321, 0.560]","-0.0299 [-0.0604, 0.0008]","0.434 [0.316, 0.563]","-0.0280 [-0.0583, -0.0085]","0.384 [0.266, 0.521]","-0.0315 [-0.0582, -0.0154]","0.333 [0.214, 0.473]"
5,lpm_fixed_all_20,"+0.0007 [-0.0305, 0.0358]","0.500 [0.371, 0.629]","+0.0032 [-0.0299, 0.0384]","0.521 [0.400, 0.643]","-0.0275 [-0.0590, 0.0013]","0.399 [0.278, 0.529]","-0.0381 [-0.0659, -0.0171]","0.319 [0.203, 0.449]"
6,lpm_fixed_l1000_20_abl_dataset_dose_time,"-0.0274 [-0.0669, 0.0075]","0.470 [0.364, 0.578]","-0.0247 [-0.0741, 0.0107]","0.491 [0.387, 0.594]","-0.0157 [-0.0348, 0.0025]","0.427 [0.304, 0.557]","-0.0204 [-0.0371, -0.0078]","0.390 [0.271, 0.513]"
7,lpm_fixed_l1000_20_abl_dataset,"-0.0054 [-0.0381, 0.0281]","0.478 [0.357, 0.598]","-0.0027 [-0.0353, 0.0313]","0.486 [0.361, 0.603]","-0.0142 [-0.0553, 0.0161]","0.478 [0.345, 0.607]","-0.0202 [-0.0501, -0.00002]","0.456 [0.332, 0.600]"
8,lpm_fixed_l1000_20_abl_dose_time,"-0.0100 [-0.0520, 0.0255]","0.486 [0.357, 0.600]","-0.0059 [-0.0546, 0.0336]","0.493 [0.359, 0.621]","-0.0058 [-0.0267, 0.0100]","0.500 [0.353, 0.621]","-0.0104 [-0.0271, 0.0027]","0.428 [0.298, 0.571]"
9,lpm_trainable_l1000_20,"+0.0049 [-0.0176, 0.0277]","0.543 [0.418, 0.667]","+0.0059 [-0.0165, 0.0278]","0.564 [0.425, 0.688]","+0.0045 [-0.0091, 0.0161]","0.593 [0.474, 0.711]","-0.0002 [-0.0123, 0.0070]","0.600 [0.470, 0.716]"



coverage_match / ablation


,method,spearman delta [95% CI],spearman win rate [95% CI],cosine delta [95% CI],cosine win rate [95% CI],mrrmse delta [95% CI],mrrmse win rate [95% CI],mae delta [95% CI],mae win rate [95% CI]
0,lpm_fixed_all_20_abl_dose_time_coverage_match,"-0.0206 [-0.0719, 0.0279]","0.464 [0.336, 0.614]","-0.0165 [-0.0689, 0.0360]","0.486 [0.357, 0.624]","-0.0397 [-0.0836, -0.0087]","0.356 [0.236, 0.499]","-0.0445 [-0.0791, -0.0212]","0.297 [0.186, 0.435]"
1,lpm_fixed_all_20_abl_dataset_coverage_match,"-0.0317 [-0.0765, 0.0115]","0.413 [0.286, 0.546]","-0.0284 [-0.0767, 0.0167]","0.435 [0.293, 0.573]","-0.0352 [-0.0730, -0.0048]","0.384 [0.264, 0.521]","-0.0406 [-0.0723, -0.0157]","0.348 [0.232, 0.479]"
2,lpm_fixed_all_20_abl_dataset_dose_time_coverag...,"-0.0315 [-0.0853, 0.0259]","0.399 [0.279, 0.536]","-0.0298 [-0.0882, 0.0256]","0.399 [0.274, 0.537]","-0.0313 [-0.0744, -0.00001]","0.320 [0.200, 0.457]","-0.0359 [-0.0766, -0.0129]","0.327 [0.203, 0.464]"
3,lpm_fixed_all_20,"-0.0048 [-0.0372, 0.0276]","0.478 [0.350, 0.609]","-0.0021 [-0.0352, 0.0330]","0.500 [0.379, 0.630]","-0.0313 [-0.0606, -0.0025]","0.370 [0.257, 0.507]","-0.0409 [-0.0678, -0.0207]","0.276 [0.171, 0.400]"
4,lpm_fixed_l1000_20,"-0.0163 [-0.0533, 0.0170]","0.449 [0.336, 0.574]","-0.0145 [-0.0507, 0.0183]","0.463 [0.348, 0.579]","-0.0294 [-0.0707, -0.0053]","0.463 [0.333, 0.585]","-0.0299 [-0.0667, -0.0103]","0.406 [0.292, 0.537]"
5,lpm_fixed_all_20_narrowed,"-0.0314 [-0.0596, -0.0011]","0.441 [0.321, 0.560]","-0.0299 [-0.0604, 0.0008]","0.434 [0.316, 0.563]","-0.0280 [-0.0583, -0.0085]","0.384 [0.266, 0.521]","-0.0315 [-0.0582, -0.0154]","0.333 [0.214, 0.473]"
6,lpm_fixed_l1000_20_abl_dataset_dose_time,"-0.0274 [-0.0669, 0.0075]","0.470 [0.364, 0.578]","-0.0247 [-0.0741, 0.0107]","0.491 [0.387, 0.594]","-0.0157 [-0.0348, 0.0025]","0.427 [0.304, 0.557]","-0.0204 [-0.0371, -0.0078]","0.390 [0.271, 0.513]"
7,lpm_fixed_l1000_20_abl_dataset,"-0.0054 [-0.0381, 0.0281]","0.478 [0.357, 0.598]","-0.0027 [-0.0353, 0.0313]","0.486 [0.361, 0.603]","-0.0142 [-0.0553, 0.0161]","0.478 [0.345, 0.607]","-0.0202 [-0.0501, -0.00002]","0.456 [0.332, 0.600]"
8,lpm_fixed_l1000_20_abl_dose_time,"-0.0100 [-0.0520, 0.0255]","0.486 [0.357, 0.600]","-0.0059 [-0.0546, 0.0336]","0.493 [0.359, 0.621]","-0.0058 [-0.0267, 0.0100]","0.500 [0.353, 0.621]","-0.0104 [-0.0271, 0.0027]","0.428 [0.298, 0.571]"
9,lpm_trainable_l1000_20,"+0.0049 [-0.0176, 0.0277]","0.543 [0.418, 0.667]","+0.0059 [-0.0165, 0.0278]","0.564 [0.425, 0.688]","+0.0045 [-0.0091, 0.0161]","0.593 [0.474, 0.711]","-0.0002 [-0.0123, 0.0070]","0.600 [0.470, 0.716]"
